In [ ]:
import numpy as np
from scipy.special import erfcinv

from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.sources import bitSource
from optic.comm.modulation import modulateGray
from optic.comm.metrics import bert
from optic.dsp.core import firFilter, pulseShape, upsample, anorm
from optic.utils import parameters, dBm2W

In [ ]:
def compute_metrics_simple(symbTx, I_Rx, bitsTx):
    BER, _ = bert(I_Rx, bitsTx)

    noise = I_Rx - symbTx
    signal_power = np.mean(symbTx**2)
    noise_power = np.mean(noise**2)

    EVM = np.sqrt(noise_power / signal_power) * 100
    Q = np.sqrt(2) * erfcinv(2 * BER)

    return {
        "BER": float(BER),
        "EVM": float(EVM),
        "Q": float(Q),
    }


def build_gmp_matrix(x: np.ndarray, P: int, M: int, L: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.complex128).reshape(-1)
    N = len(x)
    orders = np.arange(1, P + 1, 2, dtype=int)

    # Main MP terms
    xpad_mp = np.concatenate([np.zeros(M, dtype=np.complex128), x])
    cols = []
    for p in orders:
        for m in range(M + 1):
            xm = xpad_mp[M - m : M - m + N]
            cols.append(xm * (np.abs(xm) ** (p - 1)))
    Phi_mp = np.stack(cols, axis=1) if cols else np.zeros((N, 0), dtype=np.complex128)

    # Cross-envelope terms
    if L <= 0:
        return Phi_mp

    pad = M + L
    xpad = np.concatenate([np.zeros(pad, dtype=np.complex128), x])
    cols = []
    for p in orders:
        for m in range(M + 1):
            x_main = xpad[pad - m : pad - m + N]
            for l in range(1, L + 1):
                x_env = xpad[pad - m - l : pad - m - l + N]
                cols.append(x_main * (np.abs(x_env) ** (p - 1)))
    Phi_cross = np.stack(cols, axis=1) if cols else np.zeros((N, 0), dtype=np.complex128)

    return np.concatenate([Phi_mp, Phi_cross], axis=1)


class VolterraDPD:
    def __init__(self, P=7, M=5, L=2, ridge=1e-5, seq_len=8192, normalize=True):
        self.P = int(P)
        self.M = int(M)
        self.L = int(L)
        self.ridge = float(ridge)
        self.seq_len = int(seq_len)
        self.normalize = bool(normalize)
        self.w = None
        self.scale_x = 1.0
        self.scale_y = 1.0

    def _predict_1d(self, x_1d: np.ndarray) -> np.ndarray:
        x_1d = np.asarray(x_1d, dtype=np.complex128).reshape(-1)

        if self.w is None:
            return x_1d

        x_n = x_1d / (self.scale_x + 1e-12) if self.normalize else x_1d
        Phi = build_gmp_matrix(x_n, self.P, self.M, self.L)
        z_n = Phi @ self.w

        return z_n * self.scale_x if self.normalize else z_n

    def predict(self, x_input: np.ndarray, verbose: int = 0) -> np.ndarray:
        x = np.asarray(x_input)

        if x.ndim == 3 and x.shape[2] == 1:
            B, T, _ = x.shape
            out = np.zeros((B, T, 1), dtype=np.complex128)
            for b in range(B):
                out[b, :, 0] = self._predict_1d(x[b, :, 0])
            return out

        if x.ndim == 1:
            return self._predict_1d(x).reshape(-1)

        if x.ndim == 2 and x.shape[1] == 1:
            return self._predict_1d(x[:, 0]).reshape(-1, 1)

        raise ValueError(f"Unsupported x_input shape: {x.shape}")

    def fit_from_pair(self, x_input: np.ndarray, y_received: np.ndarray, verbose: int = 0) -> dict:
        x = np.asarray(x_input, dtype=np.complex128).reshape(-1)
        y = np.asarray(y_received, dtype=np.complex128).reshape(-1)

        if len(x) != len(y):
            raise ValueError(f"Length mismatch: len(x)={len(x)} vs len(y)={len(y)}")

        if self.normalize:
            self.scale_x = float(np.sqrt(np.mean(np.abs(x) ** 2) + 1e-12))
            self.scale_y = float(np.sqrt(np.mean(np.abs(y) ** 2) + 1e-12))
            x_n = x / (self.scale_x + 1e-12)
            y_n = y / (self.scale_y + 1e-12)
        else:
            x_n = x
            y_n = y

        Phi = build_gmp_matrix(y_n, self.P, self.M, self.L)
        N, K = Phi.shape
        A = Phi.conj().T @ Phi + self.ridge * np.eye(K, dtype=np.complex128)
        b = Phi.conj().T @ x_n
        self.w = np.linalg.solve(A, b)

        x_hat = Phi @ self.w
        train_nmse = 10.0 * np.log10(
            (np.mean(np.abs(x_hat - x_n) ** 2) + 1e-12)
            / (np.mean(np.abs(x_n) ** 2) + 1e-12)
        )

        info = {
            "coefficients": int(K),
            "train_nmse_db": float(train_nmse),
        }

        if verbose:
            print(info)

        return info

In [ ]:
# Parameters aligned with your original perf_sim setup

SpS = 16
M = 2
Rs = 32e9
Fs = SpS * Rs
Pi_dBm = 3
Pi = dBm2W(Pi_dBm)

# Bit source parameters
paramBits = parameters()
paramBits.nBits = 2**18
paramBits.mode = 'random'
paramBits.seed = 123

# Pulse shaping parameters
paramPulse = parameters()
paramPulse.pulseType = 'nrz'
paramPulse.SpS = SpS

# MZM parameters
paramMZM = parameters()
paramMZM.Vpi = 2
paramMZM.Vb = -paramMZM.Vpi / 2

# Linear fiber optical channel parameters
paramCh = parameters()
paramCh.L = 100
paramCh.alpha = 0.2
paramCh.D = 16
paramCh.Fc = 193.1e12
paramCh.Fs = Fs

# Photodiode parameters
paramPD = parameters()
paramPD.ideal = False
paramPD.B = Rs
paramPD.Fs = Fs
paramPD.seed = 456

In [ ]:
def optical_chain(symbTx, dpd_model=None, use_dpd=False):
    if use_dpd:
        if dpd_model is None:
            raise ValueError("dpd_model must be provided when use_dpd=True")
        x_input = symbTx.reshape(-1, 8192, 1)
        z_dpd = dpd_model.predict(x_input, verbose=0)
        drive_signal = z_dpd.flatten().real
    else:
        drive_signal = symbTx

    # upsampling
    symbolsUp = upsample(drive_signal, SpS)

    # pulse shaping
    pulse = pulseShape(paramPulse)
    sigTx = firFilter(pulse, symbolsUp)
    sigTx = anorm(sigTx)

    # optical modulation
    Ai = np.sqrt(Pi)
    sigTxo = mzm(Ai, sigTx, paramMZM)

    # linear fiber channel
    sigCh = linearFiberChannel(sigTxo, paramCh)

    # photodiode
    I_Rx = photodiode(sigCh, paramPD)

    # symbol-rate sampling
    I_Rx = I_Rx[0::SpS]

    return I_Rx


def run_system(dpd_model=None, use_dpd=False, random_seed=123):
    paramBits.seed = random_seed

    bitsTx = bitSource(paramBits)
    symbTx = modulateGray(bitsTx, M, "pam")

    I_Rx = optical_chain(symbTx, dpd_model=dpd_model, use_dpd=use_dpd)
    metrics = compute_metrics_simple(symbTx, I_Rx, bitsTx)

    return {
        "bitsTx": bitsTx,
        "symbTx": symbTx,
        "I_Rx": I_Rx,
        "BER": metrics["BER"],
        "EVM": metrics["EVM"],
        "Q": metrics["Q"],
    }

In [ ]:
# 1) Baseline: before DPD
out0 = run_system(dpd_model=None, use_dpd=False, random_seed=123)

print("Before DPD")
print(f"BER = {out0['BER']:.3e}")
print(f"EVM = {out0['EVM']:.3f} %")
print(f"Q   = {out0['Q']:.3f}")

# 2) Train Volterra DPD from the baseline pair
dpd_model = VolterraDPD(P=7, M=5, L=2, ridge=1e-5, seq_len=8192, normalize=True)

x_train = out0["symbTx"].reshape(-1, 8192, 1)
y_train = out0["I_Rx"].reshape(-1, 8192, 1)
info = dpd_model.fit_from_pair(x_train, y_train, verbose=0)

print("\nTraining")
print(f"Coefficients    = {info['coefficients']}")
print(f"Train NMSE (dB) = {info['train_nmse_db']:.2f}")

# 3) Test after DPD
out1 = run_system(dpd_model=dpd_model, use_dpd=True, random_seed=130)

print("\nAfter DPD")
print(f"BER = {out1['BER']:.3e}")
print(f"EVM = {out1['EVM']:.3f} %")
print(f"Q   = {out1['Q']:.3f}")